In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/dschettler8845/brats-2021-task1


In [3]:
!pip install -q monai nibabel SimpleITK matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.4 MB/s eta 0:00:00


In [4]:
import monai
import nibabel
import torch

print("MONAI:", monai.__version__)
print("GPU:", torch.cuda.get_device_name(0))

MONAI: 1.6.1
GPU: Tesla T4


In [5]:
import os
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/dschettler8845/brats-2021-task1")

for item in DATA_DIR.iterdir():
    print(item)

/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_00495.tar
/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar
/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_00621.tar


In [6]:
!ls /kaggle/input/datasets/dschettler8845/brats-2021-task1

BraTS2021_00495.tar  BraTS2021_00621.tar  BraTS2021_Training_Data.tar


In [7]:
import tarfile, os

TAR = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
OUT = "/kaggle/working/data"
N_CASES = 250

os.makedirs(OUT, exist_ok=True)
keep = set()
with tarfile.open(TAR, "r|") as tf:
    for m in tf:
        if not m.isfile():
            continue
        case = os.path.basename(os.path.dirname(m.name))
        if case not in keep:
            if len(keep) >= N_CASES:
                continue
            keep.add(case)
        tf.extract(m, OUT, filter="data")

print("Unpacked patients:", len(keep))

Unpacked patients: 250


In [8]:
from pathlib import Path

MODS = ["flair", "t1ce", "t1", "t2"]
files = []
for d in sorted(Path(OUT).glob("**/BraTS2021_*")):
    if not d.is_dir():
        continue
    cid = d.name
    imgs = [str(d / f"{cid}_{m}.nii.gz") for m in MODS]
    seg = d / f"{cid}_seg.nii.gz"
    if all(os.path.exists(p) for p in imgs) and seg.exists():
        files.append({"image": imgs, "label": str(seg)})

split = int(0.8 * len(files))
train_files, val_files = files[:split], files[split:]
print("Total:", len(files), "Train:", len(train_files), "Val:", len(val_files))
print(train_files[0])

Total: 249 Train: 199 Val: 50
{'image': ['/kaggle/working/data/BraTS2021_00000/BraTS2021_00000_flair.nii.gz', '/kaggle/working/data/BraTS2021_00000/BraTS2021_00000_t1ce.nii.gz', '/kaggle/working/data/BraTS2021_00000/BraTS2021_00000_t1.nii.gz', '/kaggle/working/data/BraTS2021_00000/BraTS2021_00000_t2.nii.gz'], 'label': '/kaggle/working/data/BraTS2021_00000/BraTS2021_00000_seg.nii.gz'}


In [9]:
from monai.transforms import LoadImaged
d = LoadImaged(keys=["image", "label"], ensure_channel_first=True)(train_files[0])
print("image:", d["image"].shape)
print("label:", d["label"].shape)

nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.12/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(


image: torch.Size([4, 240, 240, 155])
label: torch.Size([1, 240, 240, 155])


In [10]:
import random
from monai import transforms
from monai.data import Dataset, DataLoader
from monai.transforms import MapTransform, Randomizable

class RandModalityDropd(Randomizable, MapTransform):
    """Randomly hides 1-3 of the 4 MRI scans during training."""
    def __init__(self, keys, prob=0.5, max_drop=3):
        MapTransform.__init__(self, keys)
        self.prob, self.max_drop = prob, max_drop
    def randomize(self, data=None):
        self._do = self.R.rand() < self.prob
        n = self.R.randint(1, self.max_drop + 1)
        self._idx = self.R.choice(4, n, replace=False)
    def __call__(self, data):
        d = dict(data)
        self.randomize()
        if self._do:
            for k in self.keys:
                d[k][self._idx] = 0
        return d

random.seed(0)
train_sub = random.sample(train_files, min(200, len(train_files)))
val_sub = random.sample(val_files, min(50, len(val_files)))
ROI = (96, 96, 96)

train_tf = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"], ensure_channel_first=True),
    transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    transforms.CropForegroundd(keys=["image", "label"], source_key="image",
                               k_divisible=list(ROI), allow_smaller=True),
    transforms.RandSpatialCropd(keys=["image", "label"], roi_size=ROI, random_size=False),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    transforms.RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
    transforms.RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
    RandModalityDropd(keys="image", prob=0.5, max_drop=3),
])

val_tf = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"], ensure_channel_first=True),
    transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

train_loader = DataLoader(Dataset(data=train_sub, transform=train_tf),
                          batch_size=1, shuffle=True, num_workers=4)
val_loader = DataLoader(Dataset(data=val_sub, transform=val_tf),
                        batch_size=1, num_workers=4)

b = next(iter(train_loader))
print("train image:", b["image"].shape, "label:", b["label"].shape)

nvImageCodec is not available; skipping pydicom decoder plugin registration.
nvImageCodec is not available; skipping pydicom decoder plugin registration.


train image: torch.Size([1, 4, 96, 96, 96]) label: torch.Size([1, 3, 96, 96, 96])


In [11]:
import torch
from monai.networks.nets import SwinUNETR
from monai.losses import DiceLoss

device = torch.device("cuda")
model = SwinUNETR(in_channels=4, out_channels=3, feature_size=48,
                  use_checkpoint=True).to(device)

loss_function = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0, smooth_dr=1e-5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "million")

Parameters: 62.191941 million


In [12]:
import time, json
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import Activations, AsDiscrete, Compose
from monai.data import decollate_batch

max_epochs = 100      # test run first; change to 50 after it works
val_every = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
scaler = torch.amp.GradScaler("cuda")
dice_metric = DiceMetric(include_background=True, reduction="mean_batch")
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

history = {"epoch": [], "train_loss": [], "val_epoch": [], "dice_tc": [], "dice_wt": [], "dice_et": [], "dice_mean": []}
best_dice, best_epoch = 0.0, -1

for epoch in range(1, max_epochs + 1):
    t0 = time.time()
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        images = batch["image"].to(device)
        labels = batch["label"].float().to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            outputs = model(images)
            loss = loss_function(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        if step % 50 == 0:
            print(f"  step {step}/{len(train_loader)} loss {loss.item():.4f}")
    scheduler.step()
    epoch_loss /= len(train_loader)
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_loss)
    print(f"Epoch {epoch}/{max_epochs}  train loss {epoch_loss:.4f}  time {time.time()-t0:.0f}s")

    if epoch % val_every == 0:
        model.eval()
        with torch.no_grad(), torch.autocast("cuda"):
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].float().to(device)
                outputs = sliding_window_inference(images, ROI, 2, model, overlap=0.5)
                outputs = [post_pred(o) for o in decollate_batch(outputs)]
                dice_metric(y_pred=outputs, y=decollate_batch(labels))
        tc, wt, et = dice_metric.aggregate().tolist()
        dice_metric.reset()
        mean_dice = (tc + wt + et) / 3
        history["val_epoch"].append(epoch)
        history["dice_tc"].append(tc)
        history["dice_wt"].append(wt)
        history["dice_et"].append(et)
        history["dice_mean"].append(mean_dice)
        print(f"  VAL Dice  TC {tc:.3f}  WT {wt:.3f}  ET {et:.3f}  mean {mean_dice:.3f}")
        if mean_dice > best_dice:
            best_dice, best_epoch = mean_dice, epoch
            torch.save(model.state_dict(), "/kaggle/working/best_swinunetr.pth")
            print("  Best model saved!")

    with open("/kaggle/working/history.json", "w") as f:
        json.dump(history, f)

print(f"\nDone. Best mean Dice {best_dice:.3f} at epoch {best_epoch}")

  step 50/199 loss 0.9422
  step 100/199 loss 1.0000
  step 150/199 loss 1.0000
Epoch 1/100  train loss 0.8308  time 156s
  step 50/199 loss 0.8117
  step 100/199 loss 0.8384
  step 150/199 loss 0.4889
Epoch 2/100  train loss 0.7633  time 162s
  step 50/199 loss 0.7510
  step 100/199 loss 0.6615
  step 150/199 loss 1.0000
Epoch 3/100  train loss 0.6994  time 161s
  step 50/199 loss 0.1989
  step 100/199 loss 0.5820
  step 150/199 loss 0.3099
Epoch 4/100  train loss 0.6233  time 161s
  step 50/199 loss 0.4899
  step 100/199 loss 0.8578
  step 150/199 loss 0.6047
Epoch 5/100  train loss 0.5722  time 162s
  VAL Dice  TC 0.608  WT 0.792  ET 0.715  mean 0.705
  Best model saved!
  step 50/199 loss 0.2782
  step 100/199 loss 0.2451
  step 150/199 loss 1.0000
Epoch 6/100  train loss 0.5261  time 161s
  step 50/199 loss 0.9829
  step 100/199 loss 0.4911
  step 150/199 loss 0.9934
Epoch 7/100  train loss 0.5214  time 162s
  step 50/199 loss 0.7330
  step 100/199 loss 1.0000
  step 150/199 loss 